<div class="alert alert-block alert-info">
<b>Welcome to SynEdu!</b> This talktorial is part of <b>SynEdu</b>, a lightweight teaching series built around the <b>Syn</b> ecosystem and <b>RDKit</b> for practical, reproducible cheminformatics.
</div>

<div class="alert alert-block alert-warning">
<b>Reproducibility first</b>: Keep runtime short, prefer small datasets, and pin dependencies (e.g., via <code>env/environment.yml</code>). Save version info alongside exported figures.
</div>

<div class="alert alert-block alert-success">
<b>By the end of this notebook</b>, you will apply one L/K/R rule to a host graph using a simple subgraph match (teaching version).
</div>

# S06 · Forward apply one rule (toy graph rewrite)

**Note:** This is a teaching implementation. For chemistry-safe application, prefer SynKit when available.


## Authors and contributions

- Tieu-Long Phan, Peter Stadler group, Professur für Bioinformatik, Institut für Informatik, Universität Leipzig
- (Add contributors here)


<div class="alert alert-block alert-info">
<b>Cross-referencing</b>: When referring to another SynEdu notebook, use <b>Talktorial SXX</b> (e.g., <b>Talktorial S03</b>).
</div>


## Roadmap
- Concepts: match L, preserve K, add R
- Hands-on: apply a rule extracted in S04/S05


# Theory

Rule application requires:
1) Find a match of L inside the host graph
2) Delete L\K (the part removed)
3) Add R\K (the part added)

Chemistry-safe application also requires RDKit sanitization and valence guards.


# Practical


In [ ]:
from __future__ import annotations

from pathlib import Path
import pandas as pd
import networkx as nx

from rdkit import Chem
from rdkit.Chem import Draw

# Optional: Syn ecosystem (kept optional for Paper 1)
try:
    import synkit  # type: ignore
    HAS_SYNKit = True
except Exception:
    HAS_SYNKit = False

OUT = Path("talktorials/out")
OUT.mkdir(parents=True, exist_ok=True)

import rdkit
import networkx as nx_mod
print("RDKit:", rdkit.__version__)
print("NetworkX:", nx_mod.__version__)
print("SynKit available:", HAS_SYNKit)


In [ ]:
import json
from networkx.algorithms import isomorphism as iso

# Load one rule (from S04) and host graph from its reactant side
rule = json.loads((OUT / "S04_rule.json").read_text(encoding="utf-8"))
L = nx.node_link_graph(rule["L"])
K = nx.node_link_graph(rule["K"])
R = nx.node_link_graph(rule["R"])

df = pd.read_csv("data/reactions_mapped.csv")
row = df[df["rxn_id"] == rule["rxn_id"]].iloc[0]
react, prod = row.am_rxn_smiles.split(">>")
mR = Chem.MolFromSmiles(react)

def mol_to_mapped_graph(m: Chem.Mol) -> nx.Graph:
    G = nx.Graph()
    for a in m.GetAtoms():
        amap = a.GetAtomMapNum()
        if amap:
            G.add_node(amap, symbol=a.GetSymbol())
    for b in m.GetBonds():
        ai=b.GetBeginAtom().GetAtomMapNum(); aj=b.GetEndAtom().GetAtomMapNum()
        if ai and aj and ai in G.nodes and aj in G.nodes:
            G.add_edge(min(ai,aj), max(ai,aj), order=int(b.GetBondTypeAsDouble()))
    return G

G_host = mol_to_mapped_graph(mR)
print("Host nodes:", list(G_host.nodes(data=True)))
print("L nodes:", list(L.nodes(data=True)))


In [ ]:
def apply_rule_toy(G_host: nx.Graph, L: nx.Graph, K: nx.Graph, R: nx.Graph) -> list[nx.Graph]:
    def nm(a, b): return a.get("symbol") == b.get("symbol")
    def em(a, b): return a.get("order") == b.get("order")
    GM = iso.GraphMatcher(G_host, L, node_match=nm, edge_match=em)

    outs = []
    for m in GM.subgraph_isomorphisms_iter():
        H = G_host.copy()
        L_nodes = set(L.nodes()); K_nodes = set(K.nodes())

        for ln in (L_nodes - K_nodes):
            H.remove_node(m[ln])

        next_id = max(H.nodes, default=0) + 1
        new_map = {}
        for rn, d in R.nodes(data=True):
            if rn in K_nodes:
                continue
            new_map[rn] = next_id
            H.add_node(next_id, **d)
            next_id += 1

        def host_id(rn):
            if rn in K_nodes:
                return m[rn]
            return new_map[rn]

        for u, v, d in R.edges(data=True):
            hu, hv = host_id(u), host_id(v)
            if not H.has_edge(hu, hv):
                H.add_edge(hu, hv, **d)

        outs.append(H)
    return outs

outs = apply_rule_toy(G_host, L, K, R)
print("Applications:", len(outs))
if outs:
    print("Result nodes:", list(outs[0].nodes(data=True)))


In [ ]:
# Optional: where real SynKit would go
if HAS_SYNKit:
    print("TODO: Replace apply_rule_toy with SynKit's DPO apply (chemistry-safe).")
else:
    print("SynKit not installed; toy application shown.")


# Discussion
- You may see multiple matches → multiple candidate products.
- In practice, you also deduplicate products (canonical SMILES) and filter invalid ones.


# Quiz
1. Name three chemistry guards you would add before accepting a product.
2. Why is product deduplication needed?
3. Where would you plug in SynKit to replace the toy rewrite?


# References and further reading

*Suggested citation style:*  
* Keyword: <i>Source</i> (year) (link)

- RDKit documentation: <i>RDKit</i> (ongoing) — https://www.rdkit.org/docs/
- RDKit Book: <i>The RDKit Book</i> (ongoing) — https://www.rdkit.org/docs/Book.html
- NetworkX documentation: <i>NetworkX</i> (ongoing) — https://networkx.org/documentation/stable/
- Graphviz DOT language: <i>Graphviz</i> (ongoing) — https://graphviz.org/documentation/
